In [ ]:
from unidecode import unidecode
import pandas as pd
import re
import numpy as np

In [ ]:
# Path et colonnes

path = 'multq/rawdata/address_proprio_parsed-v1.csv'

columns = ['anrole', 'id_provinc', 'code_mun', 'code_arr', 'mat18', 'nom',
       'prenom', 'code_type', 'no_coprop', 'adresse_post', 'muni', 'code_post',
       'adresse_comp', 'date_insc', 'statut', 'num_civ', 'fraction',
       'cpde_gen', 'code_lien', 'rue', 'orient', 'num_app', 'fraction_num_app',
       'prov', 'pays', 'succ', 'cp', 'num_proprio', 'roadtype_proprio', 'streetname_proprio', 'card_proprio', 'suite_proprio', 'zip_proprio', 'cp_proprio']

In [ ]:
# Fonction de nettoyage

def clean_text(x):
    if pd.isna(x):
        return x
    x = unidecode(str(x)).lower()
    x = re.sub(r"[^A-Z0-9 ]", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

In [ ]:
# Ouvrir le fichier parsé

data = pd.read_csv(path, index_col=False, encoding="utf_8_sig")
df = pd.DataFrame(data)
print("Nombre total de lignes : ", len(df))

In [ ]:
# Nettoyage de certaines colonnes (reprise du code R)

# Nettoyer la municipalité
## Municipalité : garder uniquement le nom (supprimer QC, ON, NB, AB, BC, QUEBEC, CANADA)
df.loc[:, "muni"] = df["muni"].str.replace(
    r" (QC|ON|NB|AB|BC|QUEBEC|CANADA)$", "", regex=True)

# Normaliser la province
## # Province : remplacer QC par QUEBEC
df.loc[:, "prov"] = df["prov"].replace({
    "QC": "QUEBEC",
    "BC": "COLOMBIEBRITANNIQUE",
    "ON": "ONTARIO"})

In [ ]:
# =========================
# STRUCTURE DES TABLES PROPRIO ET MÉNAGE
# =========================

In [ ]:
proprio_tbl_temp = df.copy()
proprio_tbl_temp = proprio_tbl_temp.fillna('')

# Crée un identifiant unique pour chaque proprio en fonction de 7 variables (1) prenom, (2) nom,
# (3) num. civique, (4) num. de copropriété, (5) suite/app., (6) nom de voie et (7) code postal

proprio_tbl_temp["id_prop"] = (
    proprio_tbl_temp
    .groupby([
        "prenom", "nom", "num_proprio", "no_coprop",
        "suite_proprio", "streetname_proprio", "zip_proprio"
    ]).ngroup())

print("Nombre total de propriétaires: ",len(proprio_tbl_temp))
print("Nombre de propriétaires uniques: ",len(proprio_tbl_temp.drop_duplicates(subset=['id_prop'])))

# Crée un identifiant unique pour chaque ménage en regroupant les proprio uniques sous un ménage
# en fonction de 4 variables (1) suite, (2) num. civique, (3) nom de voie et (4) code postal

proprio_tbl_temp["id_menage"] = (
    proprio_tbl_temp
    .groupby([
        "suite_proprio", "num_proprio", "streetname_proprio", "zip_proprio"
    ]).ngroup())

print("Nombre total de propriétaires: ",len(proprio_tbl_temp))
print("Nombre de ménages uniques: ",len(proprio_tbl_temp.drop_duplicates(subset=['id_menage'])))

# Création de la table proprio
proprio_tbl_temp = proprio_tbl_temp[[
    "id_prop", "id_menage", "nom", "prenom",
    "no_coprop", "statut", "code_type", "adresse_post", "suite_proprio",
    "num_proprio", "streetname_proprio", "zip_proprio",
    "code_mun", "muni", "prov",
    "anrole", "id_provinc", "mat18"
]].copy()

## remove "_proprio"
proprio_tbl_temp.columns = [
    col.replace("_proprio", "") for col in proprio_tbl_temp.columns
]

## rename specific
proprio_tbl_temp = proprio_tbl_temp.rename(columns={
    "statut": "type_personne",
    "code_type": "type_propriete"
})

proprio_tbl_temp.to_csv("G:/Werk/CEDVS/thibault_proprio/multq/rawdata/proprio_test.csv", index=False, encoding="utf_8_sig")

# Création de la table ménage
menage_tbl_temp = proprio_tbl_temp[[
    "id_menage", "adresse_post", "suite",
    "num", "streetname", "zip"
]].copy()

## Supprime les doublons dans la table Ménage en fonction de 3 variables (1) id_menage,
## (2) num et (3) streetname
menage_tbl_temp = menage_tbl_temp.drop_duplicates(subset=['id_menage', 'num', 'streetname'], keep=False)
print("Nombre de ménages uniques dans la table Ménage: ", len(menage_tbl_temp))


menage_tbl_temp.to_csv("G:/Werk/CEDVS/thibault_proprio/multq/rawdata/menage_test.csv", index=False, encoding="utf_8_sig")


In [ ]:
# =========================
# ANALYSIS BLOCKS (reprise du code R)
# =========================

# --- propriétaires avec > 1 obs ---
v = (
    proprio_tbl_temp
    .groupby("id_prop")
    .filter(lambda x: len(x) > 1)
)

# --- nombre propriétaires distincts ---
nb_prop = proprio_tbl_temp["id_prop"].nunique()

# --- ménages avec > 1 obs ---
v2 = (
    proprio_tbl_temp
    .groupby("id_menage")
    .filter(lambda x: len(x) > 1)
)

# --- nombre ménages distincts ---
nb_menage = proprio_tbl_temp["id_menage"].nunique()

print("Nombre propriétaires distincts:", nb_prop)
print("Nombre ménages distincts:", nb_menage)